In [2]:
import pandas as pd

df = pd.read_csv('./data/titanic_procesado.csv')

print(df.shape)

df.head()

(891, 8)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,1.0,1.0,0.271174,0.125,0.0,0.014151,0.0
1,1,0.0,0.0,0.472229,0.125,0.0,0.139136,0.5
2,1,1.0,0.0,0.321438,0.000,0.0,0.015469,0.0
3,1,0.0,0.0,0.434531,0.125,0.0,0.103644,0.0
4,0,1.0,1.0,0.434531,0.000,0.0,0.015713,0.0


In [1]:
print("Kernel vivo")

Kernel vivo


In [3]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']

print(X.shape)
print(y.shape)

X.head()

(891, 7)
(891,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1.0,1.0,0.271174,0.125,0.0,0.014151,0.0
1,0.0,0.0,0.472229,0.125,0.0,0.139136,0.5
2,1.0,0.0,0.321438,0.000,0.0,0.015469,0.0
3,0.0,0.0,0.434531,0.125,0.0,0.103644,0.0
4,1.0,1.0,0.434531,0.000,0.0,0.015713,0.0


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)


(712, 7)
(179, 7)


In [5]:
X_train = X_train.values
y_train = y_train.values

X_test = X_test.values
y_test = y_test.values

print(type(X_train))
print(type(y_train))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [6]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("Todo OK")

Todo OK


In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas
from sklearn.metrics import accuracy_score

# Guardar modelos
import pickle

In [8]:
puntajes_modelos = []

estimadores = {}

mejor_modelo = None
mejor_precision = 0
mejor_estimador = None

In [10]:
modelos = {
    'Logistic Regression': {
        'model': LogisticRegression(),
        'params': {
            'model__C': [0.1],
            'model__max_iter': [1000]
        }
    },
    'Support Vector Classifier': {
        'model': SVC(),
        'params': {
            'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'model__C': [0.1, 1, 10]
        }
    },
    'Decision Tree Classifier': {
        'model': DecisionTreeClassifier(),
        'params': {
            'model__splitter': ['best', 'random'],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Random Forest Classifier': {
        'model': RandomForestClassifier(),
        'params': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4],
            'model__max_features': ['sqrt', 'log2']
        }
    },
    'Gradient Boosting Classifier': {
        'model': GradientBoostingClassifier(),
        'params': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [1, 2, 3, 4]
        }
    },
    'AdaBoost Classifier': {
        'model': AdaBoostClassifier(),
        'params': {
            'model__n_estimators': [10, 100]
        }
    },
    'K-Nearest Neighbors Classifier': {
        'model': KNeighborsClassifier(),
        'params': {
            'model__n_neighbors': [3, 5, 7]
        }
    },
    'XGBoost Classifier': {
        'model': XGBClassifier(),
        'params': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [1, 2, 3]
        }
    },
    'LGBM Classifier': {
        'model': LGBMClassifier(),
        'params': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [1, 2, 3],
            'model__learning_rate': [0.1, 0.2, 0.3],
            'model__verbose': [-1]
        }
    },
    'GaussianNB': {
        'model': GaussianNB(),
        'params': {}
    },
    'Naive Bayes Classifier': {
        'model': BernoulliNB(),
        'params': {
            'model__alpha': [0.1, 1.0, 10.0]
        }
    }
}

In [11]:
print(f"Modelos cargados: {len(modelos)}")
print(modelos.keys())

Modelos cargados: 11
dict_keys(['Logistic Regression', 'Support Vector Classifier', 'Decision Tree Classifier', 'Random Forest Classifier', 'Gradient Boosting Classifier', 'AdaBoost Classifier', 'K-Nearest Neighbors Classifier', 'XGBoost Classifier', 'LGBM Classifier', 'GaussianNB', 'Naive Bayes Classifier'])


In [12]:
for nombre, config in modelos.items():

    print(f"Entrenando {nombre}...")

    pipeline = Pipeline([
        ('model', config['model'])
    ])

    grid_search = GridSearchCV(
        pipeline,
        config['params'],
        cv=5,
        verbose=0,
        n_jobs=-1
    )

    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_train, y_train)

    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_test)

    # Calcular la precisión
    precision = accuracy_score(y_test, y_pred)

    # Guardar resultados
    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })

    estimadores[nombre] = grid_search.best_estimator_

    # Actualizar mejor modelo
    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

Entrenando Logistic Regression...
Entrenando Support Vector Classifier...
Entrenando Decision Tree Classifier...
Entrenando Random Forest Classifier...
Entrenando Gradient Boosting Classifier...
Entrenando AdaBoost Classifier...
Entrenando K-Nearest Neighbors Classifier...
Entrenando XGBoost Classifier...
Entrenando LGBM Classifier...
Entrenando GaussianNB...
Entrenando Naive Bayes Classifier...


C:\Users\Usuario\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [13]:
metricas = pd.DataFrame(puntajes_modelos).sort_values(
    'Precisión',
    ascending=False
)

print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

Rendimiento de los modelos de clasificación
                            Modelo  Precisión
1        Support Vector Classifier       0.82
4     Gradient Boosting Classifier       0.81
3         Random Forest Classifier       0.81
8                  LGBM Classifier       0.81
7               XGBoost Classifier       0.81
2         Decision Tree Classifier       0.80
5              AdaBoost Classifier       0.79
10          Naive Bayes Classifier       0.79
0              Logistic Regression       0.78
6   K-Nearest Neighbors Classifier       0.78
9                       GaussianNB       0.78
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Support Vector Classifier
Precisión: 0.82
